# Approximate token counting — is it close enough?

`ApproxCharCounter` is the fallback when a provider's exact tokenizer is
absent. The only bar it has to clear: **don't be too far off** the real
`prompt_tokens`. This notebook measures how far off it is, using the *actual
shipping* exact counters as ground truth (`make_token_counter`, not a hand-
rolled tiktoken call) over a real corpus — this repo's own source and docs,
which is the kind of text the agent actually tokenizes.

The prior under test (DeepSeek's documented offline weights, see
`tokens/base.py`): **Latin ≈ 0.3 token/char, CJK ≈ 0.6 token/char.**

In [1]:
import re
import statistics as st
from pathlib import Path

from milky_frog.domain.provider import Provider
from milky_frog.tokens import ApproxCharCounter
from milky_frog.tokens.base import _TOKENS_PER_CJK_CHAR, _TOKENS_PER_OTHER_CHAR
from milky_frog.tokens.counters import make_token_counter

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
cache = Path.home() / ".milky-frog" / "tokenizers"

approx = ApproxCharCounter()  # the estimator under test
# Ground truth = the exact counters the app actually ships, per provider.
exact_openai = make_token_counter(Provider.OPENAI, "gpt-4o", cache_dir=cache)
exact_deepseek = make_token_counter(Provider.DEEPSEEK, "deepseek-chat", cache_dir=cache)
print(f"prior: Latin {_TOKENS_PER_OTHER_CHAR} tok/char, CJK {_TOKENS_PER_CJK_CHAR} tok/char")
print(f"OpenAI exact   -> {type(exact_openai).__name__}")
print(f"DeepSeek exact -> {type(exact_deepseek).__name__}")
GROUND = [("OpenAI", exact_openai), ("DeepSeek", exact_deepseek)]

prior: Latin 0.3 tok/char, CJK 0.6 tok/char
OpenAI exact   -> TiktokenCounter
DeepSeek exact -> HFTokenizerCounter


## Corpus — segment this repo into request-sized blocks

Split source and docs at blank lines so each sample is realistic request-
sized text (not a whole file, which would wash out variance), and bucket by
script: Python code, English prose, Chinese prose.

In [2]:
CJK = re.compile(r"[\u3000-\u303f\u3400-\u4dbf\u4e00-\u9fff\uac00-\ud7a3\uff00-\uffef]")
segments: dict[str, list[str]] = {"code": [], "english": [], "chinese": []}


def add(text: str, is_code: bool) -> None:
    for block in re.split(r"\n\s*\n", text):
        block = block.strip("\n")
        if len(block) < 20:
            continue
        bucket = "code" if is_code else ("chinese" if CJK.search(block) else "english")
        segments[bucket].append(block)


for py in ROOT.rglob("src/**/*.py"):
    add(py.read_text(encoding="utf-8"), True)
for md in ["README.md", "CONTEXT.md", "CLAUDE.md", "docs/ARCHITECTURE.md"]:
    if (ROOT / md).exists():
        add((ROOT / md).read_text(encoding="utf-8"), False)

for name, blocks in segments.items():
    print(f"{name:9}: {len(blocks):5d} blocks, {sum(len(b) for b in blocks):>7} chars")

code     :  1240 blocks,  272844 chars
english  :   102 blocks,   30097 chars
chinese  :    32 blocks,    4146 chars


## How far off is the approximation?

`approx / exact` per block: **>1 over-counts** (safe — trims early), **<1
under-counts** (unsafe — risks context overflow). We want the ratio clustered
near 1. Report the median and the band 80% of blocks fall in (p10–p90).

In [3]:
def pct(xs: list[float], p: float) -> float:
    xs = sorted(xs)
    k = (len(xs) - 1) * p / 100
    f = int(k)
    return xs[f] if f + 1 >= len(xs) else xs[f] + (xs[f + 1] - xs[f]) * (k - f)


print(f"{'category':9}{'n':>5}  exact:    " + "".join(f"{name:>26}" for name, _ in GROUND))
print("-" * 75)
for cat, blocks in segments.items():
    row = f"{cat:9}{len(blocks):5d}            "
    for _, exact in GROUND:
        ratios = [approx.count_text(b) / exact.count_text(b) for b in blocks]
        row += f"{st.median(ratios):6.2f}  [{pct(ratios, 10):.2f}-{pct(ratios, 90):.2f}]      "
    print(row)

category     n  exact:                        OpenAI                  DeepSeek
---------------------------------------------------------------------------
code      1240              1.33  [1.10-1.67]        1.22  [1.00-1.53]      
english    102              1.33  [1.07-1.50]        1.26  [1.00-1.50]      
chinese     32              0.91  [0.78-1.13]        0.97  [0.89-1.18]      


## Why the per-script weights — what a flat `÷3` did

The previous prior was a single `chars/token = 3`. Against the same corpus it
was *bimodally* wrong: wasteful on Latin, dangerously low on Chinese (a Han
character is ~2 chars/token, so `÷3` systematically under-counts it).

In [4]:
def old_div3(s: str) -> int:
    return max(1, len(s) // 3)


print(f"{'category':9}{'flat ÷3 median':>18}{'unsafe%':>10}{'   new median':>16}{'unsafe%':>10}")
print("-" * 63)
for cat, blocks in segments.items():
    old = [old_div3(b) / exact_openai.count_text(b) for b in blocks]
    new = [approx.count_text(b) / exact_openai.count_text(b) for b in blocks]
    ou = 100 * sum(r < 1 for r in old) / len(blocks)
    nu = 100 * sum(r < 1 for r in new) / len(blocks)
    print(f"{cat:9}{st.median(old):18.2f}{ou:9.0f}%{st.median(new):16.2f}{nu:9.0f}%")

category     flat ÷3 median   unsafe%      new median   unsafe%
---------------------------------------------------------------


code                   1.47        2%            1.33        3%
english                1.46        3%            1.33        7%
chinese                0.66       88%            0.91       84%


## Verdict

- **Chinese** lands within ~±20% of both providers' exact counts (median
  ~0.9–1.0), vs the old `÷3`'s 0.66 — no longer "too far off".
- **Latin / code** over-count ~10–50%: the safe direction (trim early), mildly
  wasteful — and the workspace `safety_margin` already owns that slack.
- No static char-based prior nails BPE exactly; DeepSeek's own docs say the
  ratio "can vary … the actual number of tokens is based on the model's
  return." So this stays a fallback: when the exact counter is present it's
  used; usage-based calibration (v2) can fit the residual later.